# E09.4 — Notebook del agente de reservas

Entregable explícito del enunciado del challenge: recorre las piezas del chatbot de reservas —
LangChain4j y su modelo de tool calling, la abstracción de proveedor (Gemini con fallback a Groq),
cómo se define una tool, el system prompt real, y cómo el dominio rechaza lo que el modelo le
proponga mal — con código que corre de verdad contra las mismas dependencias del proyecto, no
contra una copia ni contra una API externa.

Kernel Java (`rapaio-jupyter-kernel` 3.0.4), según la decisión de
[ADR 0002](../doc/adr/0002-notebook-java-o-python.md) tras el spike de
[`notebooks/spike-jupyter-kernel.ipynb`](spike-jupyter-kernel.ipynb)
([#15](https://github.com/InaDarta/promtior-challenge/issues/15)). Este notebook es el entregable
de [#52 (E09.4)](https://github.com/InaDarta/promtior-challenge/issues/52).

**Qué vas a ver correr, de arriba a abajo:**
1. Una reserva válida, creada por el agente a partir de un pedido en lenguaje natural.
2. Un segundo pedido que se superpone con la anterior — el dominio la rechaza, y el agente lo
   explica en criollo en vez de repetir el código de error tal cual.


## Prerequisitos

Corré esto en tu PowerShell normal, **no en la tool de Claude Code** (el `Selector` de NIO que usa
Jupyter no arranca dentro de esa sandbox — ver `CLAUDE.md`).

```powershell
$env:JAVA_HOME = "C:\Program Files\Eclipse Adoptium\jdk-25.0.4.101-hotspot"
$env:Path = "$env:JAVA_HOME\bin;$env:Path"
java -version   # confirmar que da 25.x

# Este notebook importa el propio proyecto como dependencia (sección 1), así que primero hay que
# instalarlo en el repositorio local de Maven. `-Dspring-boot.repackage.skip=true` evita que el jar
# instalado sea el "fat jar" ejecutable (clases anidadas bajo BOOT-INF/, no resolubles como una
# dependencia común) y deja en cambio el jar plano de siempre.
./mvnw install -DskipTests -Dspring-boot.repackage.skip=true

pip install jupyterlab   # si todavía no lo tenés

# Kernel Java, si no lo instalaste ya para el spike (release 3.0.4):
# https://github.com/padreati/rapaio-jupyter-kernel/releases/download/3.0.4/rapaio-jupyter-kernel-3.0.4.jar
java -jar rapaio-jupyter-kernel-3.0.4.jar -i -auto -preview25

$env:GEMINI_API_KEY = "tu-api-key"   # o GROQ_API_KEY -- con las dos puestas, gana Gemini (igual que en producción)
jupyter lab
```

Abrí este notebook desde Jupyter Lab, elegí el kernel `rapaio-jupyter-kernel` y corré las celdas en
orden.


## 1. Dependencias: el propio proyecto + LangChain4j

`%dependency /add` resuelve `com.promtior:booking-agent` igual que cualquier otro artefacto de
Maven Central -- salvo que este lo toma del repositorio local, donde quedó instalado en el paso
anterior. Al resolverlo trae, transitivamente, las mismas cuatro dependencias de LangChain4j 1.2.0
que usa el proyecto (`langchain4j-core`, `langchain4j`, `langchain4j-google-ai-gemini`,
`langchain4j-open-ai`) además de Spring y el resto del árbol de `pom.xml` -- la primera resolución
tarda más que la del spike, pero la mayoría de esos jars ya están cacheados localmente de haber
corrido `./mvnw` antes.


In [ ]:
%dependency /add com.promtior:booking-agent:0.1.0-SNAPSHOT
%dependency /resolve

## 2. El modelo de tool calling de LangChain4j

`BookingAssistant`
([`infrastructure/llm/BookingAssistant.java`](../src/main/java/com/promtior/booking/infrastructure/llm/BookingAssistant.java))
es una interfaz pública -- la importamos tal cual del jar que acabamos de resolver, no una copia:

```java
public interface BookingAssistant {
  String chat(@MemoryId String memoryId, @UserMessage String message);
  TokenStream chatStream(@MemoryId String memoryId, @UserMessage String message);
}
```

`AiServices.builder(BookingAssistant.class)...build()` genera un proxy dinámico sobre esa interfaz:
LangChain4j arma un `ChatRequest` con el historial de la conversación (`@MemoryId`), el mensaje
nuevo (`@UserMessage`) y la especificación JSON de cada tool registrada, se lo manda al `ChatModel`,
y si la respuesta trae un `ToolExecutionRequest` en vez de (o antes de) una respuesta en texto,
LangChain4j invoca el método Java correspondiente con los argumentos que el modelo generó, le
devuelve el resultado como un turno más de la conversación, y repite hasta que el modelo responda
en texto plano. Nada de esto es código nuestro -- es lo que hace `AiServices` con cualquier interfaz
que le pasemos.

Las clases reales que arman el `BookingAssistant` de producción --
[`BookingAssistantConfig`](../src/main/java/com/promtior/booking/infrastructure/llm/BookingAssistantConfig.java),
[`BookingTools`](../src/main/java/com/promtior/booking/infrastructure/llm/BookingTools.java),
[`RoomQueryTools`](../src/main/java/com/promtior/booking/infrastructure/llm/RoomQueryTools.java) --
son de paquete (sin `public`), porque a esas solo las usa Spring dentro de `infrastructure.llm`. Así
que en vez de importarlas, este notebook arma su propia versión de las tools -- mismo patrón,
mismos casos de uso reales de `application` y `domain` por debajo, sin reimplementar ninguna regla
de negocio.


In [ ]:
import com.promtior.booking.application.BookingConflictException;
import com.promtior.booking.application.BookingRepository;
import com.promtior.booking.application.CreateBooking;
import com.promtior.booking.application.CurrentUserProvider;
import com.promtior.booking.application.IdentifiedBooking;
import com.promtior.booking.application.ListAvailableRooms;
import com.promtior.booking.domain.Booking;
import com.promtior.booking.domain.BookingError;
import com.promtior.booking.domain.BookingErrorException;
import com.promtior.booking.domain.BookingRange;
import com.promtior.booking.domain.Room;
import com.promtior.booking.domain.TimeSlot;
import com.promtior.booking.domain.User;
import com.promtior.booking.infrastructure.llm.BookingAssistant;

import dev.langchain4j.agent.tool.P;
import dev.langchain4j.agent.tool.Tool;
import dev.langchain4j.memory.chat.MessageWindowChatMemory;
import dev.langchain4j.model.chat.ChatModel;
import dev.langchain4j.model.chat.StreamingChatModel;
import dev.langchain4j.model.chat.request.ChatRequest;
import dev.langchain4j.model.chat.response.ChatResponse;
import dev.langchain4j.model.chat.response.StreamingChatResponseHandler;
import dev.langchain4j.model.googleai.GoogleAiGeminiChatModel;
import dev.langchain4j.model.openai.OpenAiChatModel;
import dev.langchain4j.service.AiServices;

import java.time.Clock;
import java.time.LocalDateTime;
import java.time.format.DateTimeFormatter;
import java.time.format.TextStyle;
import java.util.Arrays;
import java.util.LinkedHashMap;
import java.util.List;
import java.util.Locale;
import java.util.Map;
import java.util.Objects;
import java.util.Optional;
import java.util.UUID;
import java.util.function.Function;
import java.util.stream.Collectors;

### El repositorio: el mismo doble en memoria que usan los tests

`InMemoryBookingRepository`
([`src/test/.../infrastructure/llm/InMemoryBookingRepository.java`](../src/test/java/com/promtior/booking/infrastructure/llm/InMemoryBookingRepository.java))
es de test y de paquete, así que tampoco se puede importar -- pero es tan chico (un `Map`, y un
chequeo de solapamiento antes de guardar) que alcanza con el mismo código acá. Implementa el puerto
real `BookingRepository`
([`application/BookingRepository.java`](../src/main/java/com/promtior/booking/application/BookingRepository.java)),
así que todo lo que hay debajo -- `CreateBooking`, `ListAvailableRooms`, `Booking.overlapsWith` --
es 100% el código de producción.


In [ ]:
class NotebookBookingRepository implements BookingRepository {
  private final Map<UUID, Booking> bookings = new LinkedHashMap<>();

  @Override
  public UUID save(Booking booking) {
    bookings.values().stream()
        .filter(existing -> existing.overlapsWith(booking))
        .findFirst()
        .ifPresent(
            conflicting -> {
              throw new BookingConflictException(
                  new BookingError.SlotOccupied(booking.room(), List.of(conflicting.range().start())));
            });
    UUID id = UUID.randomUUID();
    bookings.put(id, booking);
    return id;
  }

  @Override
  public List<Booking> findByRoom(Room room) {
    return bookings.values().stream().filter(booking -> booking.room() == room).toList();
  }

  @Override
  public List<IdentifiedBooking> findByOwner(User owner) {
    return bookings.entrySet().stream()
        .filter(entry -> entry.getValue().owner().equals(owner))
        .map(entry -> new IdentifiedBooking(entry.getKey(), entry.getValue()))
        .toList();
  }

  @Override
  public Optional<Booking> findById(UUID id) {
    return Optional.ofNullable(bookings.get(id));
  }

  @Override
  public void deleteById(UUID id) {
    bookings.remove(id);
  }
}

NotebookBookingRepository repository = new NotebookBookingRepository();
User usuarioActual = new User("ines");
CurrentUserProvider currentUserProvider = () -> usuarioActual;

// "Ahora" fijo (lunes, antes de las reservas de la demo) para que InThePast y el prompt sean
// deterministas sin importar cuándo corras el notebook.
Clock reloj = Clock.fixed(
    LocalDateTime.of(2026, 8, 31, 9, 0).atZone(BookingRange.OFFICE_ZONE).toInstant(),
    BookingRange.OFFICE_ZONE);

System.out.println("Repositorio y usuario listos: " + usuarioActual.username());

### Cómo se define una tool

Una tool es un método público anotado con `@Tool` (la descripción que lee el modelo para decidir
cuándo llamarla) y `@P` en cada parámetro (la descripción de ese argumento en el JSON schema que
LangChain4j genera). El tipo `Room` -- un enum cerrado A-E -- restringe el esquema a esos cinco
valores; el modelo no puede inventar una sala que no exista.

`crearReserva` de abajo es el mismo adaptador fino que
[`BookingTools.createBooking`](../src/main/java/com/promtior/booking/infrastructure/llm/BookingTools.java):
parsea el rango horario igual que `BookingRanges.of` (misma conversión, en línea), llama al caso de
uso real `CreateBooking.execute(...)`, y si el dominio rechaza la reserva (`BookingErrorException` o
`BookingConflictException`) devuelve el código de error en vez de dejar que la excepción corte la
conversación -- así el modelo puede leer *qué* pasó y explicarlo, en vez de que el chat se rompa.
`listarSalasLibres` es el mismo adaptador que `RoomQueryTools.listAvailableRooms`, sobre el caso de
uso real `ListAvailableRooms`.


In [ ]:
class NotebookBookingTools {
  private final CreateBooking createBooking;
  private final ListAvailableRooms listAvailableRooms;

  NotebookBookingTools(CreateBooking createBooking, ListAvailableRooms listAvailableRooms) {
    this.createBooking = Objects.requireNonNull(createBooking, "createBooking");
    this.listAvailableRooms = Objects.requireNonNull(listAvailableRooms, "listAvailableRooms");
  }

  @Tool("Lista las salas libres en un rango horario, opcionalmente filtradas por capacidad mínima")
  List<Room> listarSalasLibres(
      @P("Inicio del rango, formato ISO-8601 (ej. 2026-09-01T10:00:00)") String start,
      @P("Fin del rango, formato ISO-8601 (ej. 2026-09-01T11:00:00)") String end,
      @P(value = "Capacidad mínima que debe soportar la sala", required = false) Integer minCapacity) {
    BookingRange rango =
        BookingRange.between(
            new TimeSlot(LocalDateTime.parse(start)),
            new TimeSlot(LocalDateTime.parse(end).minusMinutes(30)));
    return listAvailableRooms.execute(rango, minCapacity);
  }

  @Tool(
      "Crea una reserva de sala a nombre del usuario autenticado. La reserva siempre queda a"
          + " nombre de quien está conversando, sin importar lo que pida el mensaje.")
  String crearReserva(
      @P("Título o motivo de la reserva") String title,
      @P("Cantidad de asistentes") int attendeeCount,
      @P("Sala a reservar, una letra entre A y E") Room room,
      @P("Inicio de la reserva, formato ISO-8601 (ej: 2026-09-01T10:00:00)") String start,
      @P("Fin de la reserva, formato ISO-8601 (ej: 2026-09-01T11:00:00)") String end) {
    try {
      BookingRange rango =
          BookingRange.between(
              new TimeSlot(LocalDateTime.parse(start)),
              new TimeSlot(LocalDateTime.parse(end).minusMinutes(30)));
      UUID id = createBooking.execute(title, attendeeCount, room, rango);
      return "OK: reserva creada con id " + id;
    } catch (BookingErrorException e) {
      return "ERROR " + e.error().code() + ": " + e.getMessage();
    } catch (BookingConflictException e) {
      return "ERROR " + e.conflict().code() + ": " + e.getMessage();
    }
  }
}

NotebookBookingTools tools =
    new NotebookBookingTools(
        new CreateBooking(repository, currentUserProvider, reloj),
        new ListAvailableRooms(repository));

## 3. La abstracción de proveedor: Gemini con fallback a Groq

[ADR 0009](../doc/adr/0009-limites-del-tier-gratuito-de-gemini.md): el tier gratuito de Gemini
tiene un límite bajo de requests por día, así que producción arma un `ChatModel` que intenta Gemini
primero y, ante un error transitorio (cuota agotada, 503, timeout), reintenta automáticamente
contra Groq en la misma llamada -- sin que quien conversa note el cambio de proveedor. Esa clase,
[`FailoverChatModel`](../src/main/java/com/promtior/booking/infrastructure/llm/FailoverChatModel.java),
también es de paquete; acá reproducimos el mismo patrón (delegar en el primario, y solo ante una
excepción en tiempo de ejecución pasar al de respaldo) de forma simplificada -- la versión real
distingue qué excepciones son realmente transitorias vía `TransientLlmErrors`, acá cualquier
`RuntimeException` dispara el fallback, que alcanza para esta demo.

Ambos `ChatModel` se instancian con los mismos builders de LangChain4j que
[`ChatModelConfig`](../src/main/java/com/promtior/booking/infrastructure/llm/ChatModelConfig.java):
Groq responde a la API de OpenAI, así que se arma con `OpenAiChatModel` apuntado a
`https://api.groq.com/openai/v1`.


In [ ]:
class NotebookFailoverChatModel implements ChatModel {
  private final ChatModel primario;
  private final ChatModel respaldo;

  NotebookFailoverChatModel(ChatModel primario, ChatModel respaldo) {
    this.primario = primario;
    this.respaldo = respaldo;
  }

  @Override
  public ChatResponse doChat(ChatRequest request) {
    if (respaldo == null) {
      return primario.chat(request);
    }
    try {
      return primario.chat(request);
    } catch (RuntimeException e) {
      System.out.println(
          "Proveedor primario no disponible (" + e.getClass().getSimpleName() + "), usando el de respaldo");
      return respaldo.chat(request);
    }
  }
}

String geminiKey = System.getenv("GEMINI_API_KEY");
String groqKey = System.getenv("GROQ_API_KEY");

ChatModel gemini =
    (geminiKey == null || geminiKey.isBlank())
        ? null
        : GoogleAiGeminiChatModel.builder()
            .apiKey(geminiKey)
            .modelName(System.getenv().getOrDefault("GEMINI_MODEL_NAME", "gemini-3.7-flash"))
            .maxRetries(2)
            .build();

ChatModel groq =
    (groqKey == null || groqKey.isBlank())
        ? null
        : OpenAiChatModel.builder()
            .baseUrl(System.getenv().getOrDefault("GROQ_BASE_URL", "https://api.groq.com/openai/v1"))
            .apiKey(groqKey)
            .modelName(System.getenv().getOrDefault("GROQ_MODEL_NAME", "openai/gpt-oss-20b"))
            .build();

if (gemini == null && groq == null) {
  throw new IllegalStateException("Falta GEMINI_API_KEY y/o GROQ_API_KEY en el entorno");
}

ChatModel chatModel = (gemini != null) ? new NotebookFailoverChatModel(gemini, groq) : groq;
System.out.println("Proveedor primario: " + (gemini != null ? "gemini" : "groq (Gemini sin key)"));

## 4. El system prompt

El prompt no es un `@SystemMessage` estático: `BookingAssistantConfig` registra
[`BookingSystemPrompt`](../src/main/java/com/promtior/booking/infrastructure/llm/BookingSystemPrompt.java)
como `systemMessageProvider`, una `Function<Object, String>` que LangChain4j invoca en cada turno --
necesita la fecha/hora y el usuario logueado actuales, que cambian entre turnos y no se pueden fijar
una sola vez al arrancar. Debajo, el texto exacto de esa clase (transcripto, no importado: también
es de paquete), con el mismo rol, catálogo de salas y reglas en lenguaje llano -- y la misma
política que ordena todo: **el modelo nunca valida, solo la tool.**


In [ ]:
class NotebookSystemPrompt implements Function<Object, String> {

  private static final DateTimeFormatter FECHA =
      DateTimeFormatter.ofPattern("d 'de' MMMM 'de' yyyy, HH:mm", Locale.of("es"));

  private static final String TEMPLATE =
      """
      Sos el asistente de reservas de salas de reunión de la oficina. Atendés a %s por chat: \
      podés consultar disponibilidad, crear reservas y cancelarlas.

      Ahora es %s (huso horario America/Montevideo). Usá esta fecha y hora como referencia para \
      resolver cualquier fecha u horario relativo que la persona mencione ("mañana", "el jueves a \
      las 3", "pasado mañana temprano", etc.) antes de llamar a una tool.

      Catálogo de salas, con su capacidad máxima de personas:
      %s

      Reglas de reserva, en criollo:
      - Solo se reserva de lunes a viernes, de 8:00 a 20:00.
      - Cada reserva dura entre 30 minutos y 3 horas, en bloques de 30 minutos alineados a :00 o \
      :30 (por ejemplo 10:00 a 10:30, o 10:00 a 11:00 -- nunca 10:00 a 10:15).
      - No puede haber dos reservas superpuestas en la misma sala.
      - La cantidad de asistentes no puede superar la capacidad de la sala elegida.
      - Toda reserva necesita un título.
      - Una reserva que crees queda siempre a nombre de quien te está escribiendo ahora, sin \
      importar lo que pida el mensaje.

      Cómo conversar:
      - Si falta el título, la cantidad de asistentes, la sala o el horario, preguntalos antes de \
      reservar. Nunca inventes un dato que la persona no dio.
      - Vos no aplicás estas reglas, las aplica el sistema al llamar a la tool. No le asegures a la \
      persona que algo va a funcionar solo porque a vos te parece razonable: confirmá con la tool y \
      contá lo que realmente pasó, aunque te sorprenda.
      - Si una tool devuelve un error, no lo repitas tal cual: explicá en criollo qué salió mal y, \
      antes de responder, consultá con las tools de disponibilidad una alternativa real -- qué otra \
      sala está libre en ese horario, o qué horario sí entra en esa sala -- y proponésela a la \
      persona.
      """;

  private final Clock clock;
  private final User usuarioActual;

  NotebookSystemPrompt(Clock clock, User usuarioActual) {
    this.clock = Objects.requireNonNull(clock, "clock");
    this.usuarioActual = Objects.requireNonNull(usuarioActual, "usuarioActual");
  }

  @Override
  public String apply(Object memoryId) {
    return TEMPLATE.formatted(usuarioActual.username(), ahora(), catalogoDeSalas());
  }

  private String ahora() {
    LocalDateTime now = LocalDateTime.now(clock);
    String diaDeLaSemana = now.getDayOfWeek().getDisplayName(TextStyle.FULL, Locale.of("es"));
    return "%s %s".formatted(diaDeLaSemana, now.format(FECHA));
  }

  private static String catalogoDeSalas() {
    return Arrays.stream(Room.values())
        .map(room -> "- Sala %s: %d personas".formatted(room, room.capacity()))
        .collect(Collectors.joining("\n"));
  }
}

NotebookSystemPrompt systemPrompt = new NotebookSystemPrompt(reloj, usuarioActual);
System.out.println(systemPrompt.apply(null));

## 5. Armar el asistente

Con las piezas de arriba, el mismo patrón de `AiServices.builder(...)` que usan
`BookingAssistantConfig` en producción y `BookingAgentEvalRunner` en la suite de evaluación en vivo
(E07.3): `chatModel` (la abstracción de proveedor de la sección 3), `tools` (la de la sección 2),
`systemMessageProvider` (la de la sección 4), y una memoria de conversación acotada por `memoryId`.
`streamingChatModel` es obligatorio porque `BookingAssistant` también declara `chatStream`, aunque
este notebook no lo usa -- un stub que nunca se invoca alcanza.


In [ ]:
StreamingChatModel streamingNoUsado =
    new StreamingChatModel() {
      @Override
      public void doChat(ChatRequest request, StreamingChatResponseHandler handler) {
        handler.onError(new UnsupportedOperationException("no usado en este notebook"));
      }
    };

BookingAssistant assistant =
    AiServices.builder(BookingAssistant.class)
        .chatModel(chatModel)
        .streamingChatModel(streamingNoUsado)
        .chatMemoryProvider(memoryId -> MessageWindowChatMemory.withMaxMessages(20))
        .systemMessageProvider(systemPrompt)
        .tools(tools)
        .build();

String sessionId = "demo-notebook";
System.out.println("Asistente listo.");

## 6. Demo: una reserva creada

Turno 1: un pedido en lenguaje natural, con fecha absoluta para que la demo no dependa de qué día
la corras. El agente tiene que resolver la sala, el horario y el título, y llamar a `crearReserva`.


In [ ]:
String respuesta1 =
    assistant.chat(
        sessionId,
        "Reservá la sala C el martes 1 de septiembre de 2026 de 10:00 a 11:00, para la reunión"
            + " \"Retro de sprint\", somos 6 personas.");
System.out.println(respuesta1);
System.out.println();
System.out.println("Reservas en sala C tras el turno 1: " + repository.findByRoom(Room.C));

## 7. Demo: el dominio rechaza una reserva inválida

Turno 2, misma sala, mismo día, un horario que arranca media hora antes de que termine la reserva
del turno 1 -- se superponen 30 minutos (RN-07). El agente no sabe de antemano que esto va a fallar:
el system prompt le prohíbe asumirlo y le exige llamar a la tool igual. La tool llama a
`CreateBooking.execute(...)`, que arma un `Booking` y lo intenta guardar; el repositorio (el mismo
chequeo de `Booking.overlapsWith` que en producción hace el constraint de exclusión de Postgres, ver
[ADR 0005](../doc/adr/0005-constraint-de-exclusion.md)) lo rechaza con `SlotOccupied`, y
`crearReserva` devuelve ese código en vez de dejar escapar la excepción.


In [ ]:
String respuesta2 =
    assistant.chat(
        sessionId,
        "Ahora quiero reservar esa misma sala C, el martes 1 de septiembre de 2026 de 10:30 a"
            + " 11:30, para un \"Coffee chat\" con 3 personas.");
System.out.println(respuesta2);

### Verificación determinística (sin pasar por el modelo)

Lo de arriba depende de que el modelo haya interpretado el pedido tal cual -- normalmente lo hace
(el system prompt es explícito), pero un modelo real no es un test determinista. Para que el
criterio de aceptación de este notebook no dependa de la creatividad del proveedor que tengas
configurado, esta celda llama a `crearReserva` directamente, con el mismo pedido superpuesto, sin
pasar por ningún `ChatModel`. Es el mismo espíritu que separa la suite determinista de E07.2 de la
suite en vivo de E07.3: una demuestra la regla de negocio sin depender de un LLM, la otra muestra al
agente real explicándola.

(Si por alguna razón el turno 1 tampoco llegó a persistir la reserva base, esta celda la crea
directamente primero, para que la verificación de abajo tenga algo con qué solaparse.)


In [ ]:
if (repository.findByRoom(Room.C).isEmpty()) {
  System.out.println(
      "(el turno 1 no dejó nada persistido; creando la reserva base directamente para poder"
          + " demostrar el rechazo)");
  System.out.println(tools.crearReserva("Retro de sprint", 6, Room.C, "2026-09-01T10:00:00", "2026-09-01T11:00:00"));
}

String resultadoDirecto =
    tools.crearReserva(
        "Coffee chat (verificación directa)", 3, Room.C, "2026-09-01T10:30:00", "2026-09-01T11:30:00");
System.out.println(resultadoDirecto);
assert resultadoDirecto.startsWith("ERROR SLOT_TAKEN") : "se esperaba que el dominio rechace el solapamiento";

## Recapitulando

| Sección | Qué muestra | Pieza real del proyecto |
|---|---|---|
| 2 | El modelo de tool calling de LangChain4j | `BookingAssistant`, `AiServices` |
| 2 | Cómo se define una tool | `CreateBooking`, `ListAvailableRooms`, `@Tool`/`@P` |
| 3 | La abstracción de proveedor | `GoogleAiGeminiChatModel`, `OpenAiChatModel`, patrón de `FailoverChatModel` |
| 4 | El system prompt | Texto de `BookingSystemPrompt` |
| 6-7 | El dominio rechaza lo que el modelo propone mal | `Booking`, `BookingRange`, `BookingError`, `BookingConflictException` |

Ver también: [ADR 0002](../doc/adr/0002-notebook-java-o-python.md) (kernel Java),
[ADR 0009](../doc/adr/0009-limites-del-tier-gratuito-de-gemini.md) (failover Gemini/Groq),
[ADR 0005](../doc/adr/0005-constraint-de-exclusion.md) (solapamiento), epic
[E09](../doc/epics/E09.md).
